In [ ]:
# Setup and helper functions moved to external module
from equations_file import *

# If you change the external module and want live reload during development, uncomment:
# %load_ext autoreload
# %autoreload 2

In [ ]:
save_dir = "/Users/simoncorrera/Desktop/MQ NGC3201 project copy/sequoia paper matieral/Thesis only figures"

In [ ]:
with fitsio.FITS(galah_Gaia_fits) as hdul1:
    data_1 = hdul1[1].read()
galah_Gaia_raw = pd.DataFrame(data_1)
galah_Gaia_raw = ensure_native_endian(galah_Gaia_raw)

In [ ]:
galah_Gaia = galah_Gaia_raw[(galah_Gaia_raw['snr_px_ccd3'] > 30)       # Kushniruk 2026 also has cuts in log g and temp. I wonder if I should as well
                        & (galah_Gaia_raw['flag_sp'] == 0)
                        & (galah_Gaia_raw['flag_fe_h'] == 0)
                        & (galah_Gaia_raw['e_fe_h'] < 0.1)
                        & (galah_Gaia_raw['flag_sp_fit'] == 0)
                        & (galah_Gaia_raw['flag_red'] == 0)
                        & (galah_Gaia_raw['ruwe'] < 1.4)
                        & (galah_Gaia_raw['logg'] < 3.5)
                        & (galah_Gaia_raw['teff'] > 4000)
                        & (galah_Gaia_raw['teff'] < 6500)
                        # & (galah_Gaia_raw['flag_mg_fe'] == 0)
                        # & (galah_Gaia_raw['flag_na_fe'] == 0)
                        # & (galah_Gaia_raw['flag_cu_fe'] == 0)
                        & (galah_Gaia_raw['fe_h'] < -0.8)
                        # & (galah_Gaia_raw['fe_h'] < -1.24)  # the 16th and 84th percentile spreads of all flagged elements sequoia Fe/H: -1.24, -1.72
                        # & (galah_Gaia_raw['fe_h'] > -1.72) # the 16th and 84th percentile spreads of only Fe/h flagged elements sequoia Fe/H: -1.28, -2.01
                        # & (galah_Gaia_raw['ti_fe'] > 0.25)
                        
                        & (galah_Gaia_raw['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main',
                                                               'k2_hermes', 'galah_phase2', 'tess_hermes'
                                                               ]))
                        & (galah_Gaia_raw['radial_velocity'].notnull())
                        & (galah_Gaia_raw['ra_gaia'].notnull())
                        & (galah_Gaia_raw['dec_gaia'].notnull())
                        & (galah_Gaia_raw['pmra_gaia'].notnull())
                        & (galah_Gaia_raw['pmdec_gaia'].notnull())
                        & (galah_Gaia_raw['parallax_gaia'] > 0)
                        
    ]

print(f"Number of stars after cuts: {len(galah_Gaia)}")

In [ ]:
NGC3201_galah = pd.merge( galah_Gaia, Vmans_NGC3201_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC5139_galah = pd.merge( galah_Gaia, Vmans_NGC5139_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')
NGC1851_galah = pd.merge( galah_Gaia, Vmans_NGC1851_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')   
NGC0104_galah = pd.merge( galah_Gaia, Vmans_NGC0104_high_prob, left_on = 'gaiadr3_source_id', right_on = 'source_id', how = 'inner')

galah_Kushniruk_GSE = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-35)**2) / (17.5**2)) + (((galah_Gaia['jphi'] + 130)**2) / (250**2)) < 1)
                                #  & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                 & (galah_Gaia_raw['flag_mg_fe'] == 0)
                                 & (galah_Gaia_raw['flag_na_fe'] == 0)
                                 & (galah_Gaia_raw['flag_cu_fe'] == 0)
                                 & ((galah_Gaia_raw['mg_fe'] - galah_Gaia_raw['cu_fe']) - ((1.06923) * galah_Gaia_raw['na_fe']) > 0.40150)
                                 ]

# Kushniruk et al . 2026 https://doi.org/10.1051/0004-6361/202451201
galah_Kushniruk_Thamnos1 = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])- 13.5)**2) / (4.0**2)) + (((galah_Gaia['jphi'] + 600)**2) / (350**2)) < 1)
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                        & (galah_Gaia_raw['fe_h'] < -0.8)
                                      ]

galah_Kushniruk_Thamnos2 = galah_Gaia[((((np.sqrt(galah_Gaia['jr'])-8.6)**2) / (3.5**2)) + (((galah_Gaia['jphi'] + 1184)**2) / (175**2)) < 1)
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                        # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                                        # & (galah_Gaia['survey_name'].isin(['galah_faint', 'galah_bright', 'galah_main']))
                                        & (galah_Gaia_raw['fe_h'] < -0.8)
                                      ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_GSE = galah_Gaia[GSE_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                            ]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_Sequoia = galah_Gaia[Sequoia_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                            # & (galah_Gaia_raw['fe_h'] > -1.3)

                            # & (galah_Gaia_raw['ca_fe'] < 0.37)
]

# Feuillet et al. 2021 https://doi.org/10.1093/mnras/stab2614
galah_Gaia_Sequoia_p_thamnos = galah_Gaia[Sequoia_cuts_Diane_2021(galah_Gaia)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))

                            # & (galah_Gaia_raw['y_fe'] < 0.8)
]

galah_Gaia_Halo = galah_Gaia[(np.sqrt((galah_Gaia['vphi']-230)**2 + (galah_Gaia['vr'])**2 + (galah_Gaia['vz'])**2) > 230) # Koppelman et al. 2019  https://doi.org/10.1051/0004-6361/201936738
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC5139_galah['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id'])) 
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id'])) 
                            # & (galah_Gaia_raw['fe_h'] < -1.1)
                            # & (galah_Gaia_raw['fe_h'] > -1.9)
                            & (galah_Gaia_raw['fe_h'] < -0.8)
                             ]


In [ ]:
Halo_Pradosh_galah_Gaia = galah_Gaia[
                                            (galah_Gaia['fe_h'] < -0.8)&
                                            (galah_Gaia['ti_fe'] > 0.25)
                                            & (galah_Gaia['flag_ti_fe'] == 0)
                                            & (galah_Gaia['Lz'] > -1500)
                                            & (galah_Gaia['Lz'] < 1500)
                                            & (galah_Gaia['ecc_gaia'] > 0.5)
                                            & (galah_Gaia['ecc_gaia'] < 0.8)
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                                            & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                            # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]

thickdisk_galah_Gaia = galah_Gaia[(galah_Gaia['fe_h'] < -0.8)
                                                 & (galah_Gaia['ti_fe'] > 0.25)
                                                 & (galah_Gaia['Lz'] > 0)
                                                 & (galah_Gaia['ecc_gaia'] < 0.5)
                                                #  & (~galah_Gaia['gaiadr3_source_id'].isin(NGC3201_galah['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_GSE['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_Sequoia['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos1['gaiadr3_source_id']))
                                                & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Kushniruk_Thamnos2['gaiadr3_source_id']))
                                                # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC3201_new_ratios['gaiadr3_source_id']))
                                                # & (~galah_Gaia['gaiadr3_source_id'].isin(galah_Gaia_NGC5139_new_ratios['gaiadr3_source_id']))
                                            ]
print(len(Halo_Pradosh_galah_Gaia))
print(len(thickdisk_galah_Gaia))

galah_Gaia_Halo = Halo_Pradosh_galah_Gaia

In [ ]:
# print(f'len(NGC3201_galah): {len(NGC3201_galah)}')
# print(f'len(NGC5139_galah): {len(NGC5139_galah)}')
# print(f'len(NGC1851_galah): {len(NGC1851_galah)}')
# print(f'len(NGC0104_galah): {len(NGC0104_galah)}')
# print(f'len(galah_Kushniruk_GSE): {len(galah_Kushniruk_GSE)}')
print(f'len(galah_Kushniruk_Thamnos1): {len(galah_Kushniruk_Thamnos1)}')
print(f'len(galah_Kushniruk_Thamnos2): {len(galah_Kushniruk_Thamnos2)}')
print(f'len(galah_Gaia_GSE): {len(galah_Gaia_GSE)}')
print(f'len(galah_Gaia_Sequoia): {len(galah_Gaia_Sequoia)}')
print(f'len(galah_Gaia_Sequoia_p_thamnos): {len(galah_Gaia_Sequoia_p_thamnos)}')
print(f'len(galah_Gaia_Halo): {len(galah_Gaia_Halo)}')
print(f'galah_Gaia_new_ratios: {len(galah_Gaia)}')

In [ ]:

element_list_partial_small_galah_7 = [ 'fe', 'cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_2 = ['cu', 'ca', 'mn', 'ti', 'y', 'ni', 'zn', 'v', 'ba','na', 'nd', 'si', 'k', 'sc', 'cr', 'co']
element_list_partial_small_galah_7_3 = ['fe', 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']
element_list_partial_small_galah_7_4 = ['fe', 'si', 'nd', 'na', 'mg', 'k', 'ca', 'sc', 'ti', 'v', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba']
element_list_partial_small_galah_7_5 = [ 'v', 'sc', 'na', 'mg', 'si', 'k', 'ca', 'ti', 'cr', 'mn', 'co', 'ni', 'cu', 'zn', 'y', 'ba', 'nd']  # now I wonder if I could use sm_fe, o_fe, 
element_list_partial_small_galah_10 = ['fe', 'v', 'sc', 'si', 'ca', 'mn', 'na', 'nd']                                                             # and/or la_fe
element_list_partial_small_galah_11 = ['y', 'ba', 'nd', 'eu']                                                                                 # also la and sm would bring me to (50, 45, or 44combined)/65 sequoia stars otherwise
element_list_partial_small_galah_16 =  ["na", "mg", "ca", "ti", "cr" ,"mn", "ni", 'cu', 'y', "ba", "nd"] #This is what I used in my Poster
element_list_partial_small_galah_17 =  ['fe', "na", "mg", "ca", "ti", "cr" ,"mn", "ni", 'cu', 'y', "ba", "nd"]
element_list_partial_small_galah_18 =  ['fe', "na", "mg", 'si', "ca", "ti", "mn", "ni", 'cu', 'y', "ba", "nd"]


ASA_Plots_ratios = ['fe', 'si', 'ca', 'ti', 'mn', 'ni', 'y', 'nd']
S_VS_G_best_elms = ['na', 'mn', 'ti', 'cr']

# element_list = S_VS_G_best_elms
element_list = element_list_partial_small_galah_18


galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia, element_list)

galah_Gaia_new_ratios = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0] # this ontains a full catalog from galah but now it includes ratios for Ca, Si, Ti, Ni, Zr, Ce, and Nd relative to eachother: Pradosh did recomend taking out
                                                                 # Si, and Ni whitch I can do by changing "element_list_partial_large" to element_list_partial_small
galah_gaia_new_element_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]     # this just lists the new columns ratios so I can more easily call them
galah_gaia_new_element_ratio_errors_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[2] 

galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_GSE, element_list)
galah_Gaia_GSE_new_ratios = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_GSE, element_list)
galah_Gaia_Kushniruk_GSE_new_ratios = galah_Gaia_Kushniruk_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Sequoia, element_list)
galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Sequoia_p_thamnos, element_list)
galah_Gaia_Sequoia_p_Thamnos_new_ratios = galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC3201_galah, element_list)
# galah_Gaia_NGC3201_new_ratios = galah_Gaia_NGC3201_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC5139_galah, element_list)
galah_Gaia_NGC5139_new_ratios = galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos1, element_list)
galah_Gaia_Thamnos1_new_ratios = galah_Gaia_Thamnos1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Kushniruk_Thamnos2, element_list)
galah_Gaia_Thamnos2_new_ratios = galah_Gaia_Thamnos2_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(NGC1851_galah, element_list)
# galah_Gaia_NGC1851_new_ratios = galah_Gaia_NGC1851_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_Halo, element_list)
galah_Gaia_Halo_new_ratios = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(Halo_Pradosh_galah_Gaia, element_list)
Halo_Pradosh_galah_Gaia_new_ratios = Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

# galah_Gaia_else_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_else, element_list)
# galah_Gaia_else_new_ratios = galah_Gaia_else_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_raw_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah(galah_Gaia_raw, element_list)
galah_Gaia_raw_new_ratios = galah_Gaia_raw_new_ratios_catolog_ratiolist_ratioerrorlist[0]


'----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------'

# element_list_full_galah_almost = ["n", "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", "y", "zr", "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.
element_list_full_galah_almost = ['fe', 'c', 'n', "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", 
                                  'rb', 'sr', 
                                  "y", "zr", 
                                  'mo', 'ru',
                                  "ba", "la", "ce", "nd", "sm", "eu"]# there is nn_li in galah (nural network Li/Fe) but I dont want to include that right now.


element_list_nessessary_flags = element_list_full_galah_almost

galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia, element_list_nessessary_flags)
galah_Gaia_new_ratios_nessessary_flags_ratio_list = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[1]
galah_Gaia_new_ratios_nessessary_flags = galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Sequoia, element_list_nessessary_flags)
galah_Gaia_Sequoia_new_ratios_nessessary_flags = galah_Gaia_Sequoia_new_ratios_catolog_ratiolist_ratioerrorlist[0]
 
galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Sequoia_p_thamnos, element_list_nessessary_flags)
galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags = galah_Gaia_Sequoia_p_Thamnos_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_GSE, element_list_nessessary_flags)
galah_Gaia_GSE_new_ratios_nessessary_flags = galah_Gaia_GSE_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos_1_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Kushniruk_Thamnos1, element_list_nessessary_flags)
galah_Gaia_Thamnos_1_new_ratios_nessessary_flags = galah_Gaia_Thamnos_1_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_Thamnos_2_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Kushniruk_Thamnos2, element_list_nessessary_flags)
galah_Gaia_Thamnos_2_new_ratios_nessessary_flags = galah_Gaia_Thamnos_2_new_ratios_catolog_ratiolist_ratioerrorlist[0]


galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(galah_Gaia_Halo, element_list_nessessary_flags)
galah_Gaia_Halo_new_ratios_nessessary_flags = galah_Gaia_Halo_new_ratios_catolog_ratiolist_ratioerrorlist[0]

Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(Halo_Pradosh_galah_Gaia, element_list_nessessary_flags)
Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags = Halo_Pradosh_galah_Gaia_new_ratios_catolog_ratiolist_ratioerrorlist[0]

galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist = make_elm_ratio_combinations_galah_only_flag_nessessary_columns(NGC5139_galah, element_list_nessessary_flags)
galah_Gaia_NGC5139_new_ratios_nessessary_flags = galah_Gaia_NGC5139_new_ratios_catolog_ratiolist_ratioerrorlist[0]



In [ ]:
print(f'len(galah_Gaia_Sequoia_new_ratios): {len(galah_Gaia_Sequoia_new_ratios)}')
print(f'len(galah_Gaia_Sequoia_p_Thamnos_new_ratios): {len(galah_Gaia_Sequoia_p_Thamnos_new_ratios)}')
print(f'len(galah_Gaia_GSE_new_ratios): {len(galah_Gaia_GSE_new_ratios)}')
print(f'len(galah_Gaia_Halo_new_ratios): {len(galah_Gaia_Halo_new_ratios)}')
print(f'len(galah_Gaia_else_new_ratios): {len(galah_Gaia_new_ratios)}')

In [ ]:
Xs = ['c', 'n', "o", "na", "mg", "al", "si", "k", "ca", "sc" ,"ti", "v", "cr" ,"mn", "co", "ni", "cu", "zn", 
                                  'rb', 'sr', 
                                  "y", "zr", 
                                  'mo', 'ru',
                                  "ba", "la", "ce", "nd", "sm", "eu"]
for X in Xs:
    X_fe = X + '_fe'
    X_H = X + '_h'

    galah_Gaia_new_ratios_nessessary_flags[X_H] = galah_Gaia_new_ratios_nessessary_flags[X_fe] + galah_Gaia_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Sequoia_new_ratios_nessessary_flags[X_H] = galah_Gaia_Sequoia_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Sequoia_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags[X_H] = galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Sequoia_p_Thamnos_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_GSE_new_ratios_nessessary_flags[X_H] = galah_Gaia_GSE_new_ratios_nessessary_flags[X_fe] + galah_Gaia_GSE_new_ratios_nessessary_flags['fe_h']
    Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags[X_H] = Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags[X_fe] + Halo_Pradosh_galah_Gaia_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Thamnos_1_new_ratios_nessessary_flags[X_H] = galah_Gaia_Thamnos_1_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Thamnos_1_new_ratios_nessessary_flags['fe_h']
    galah_Gaia_Thamnos_2_new_ratios_nessessary_flags[X_H] = galah_Gaia_Thamnos_2_new_ratios_nessessary_flags[X_fe] + galah_Gaia_Thamnos_2_new_ratios_nessessary_flags['fe_h']
   
    
    galah_Gaia_new_ratios[X_H] = galah_Gaia_new_ratios[X_fe] + galah_Gaia_new_ratios['fe_h']
    galah_Gaia_Sequoia_new_ratios[X_H] = galah_Gaia_Sequoia_new_ratios[X_fe] + galah_Gaia_Sequoia_new_ratios['fe_h']
    galah_Gaia_Sequoia_p_Thamnos_new_ratios[X_H] = galah_Gaia_Sequoia_p_Thamnos_new_ratios[X_fe] + galah_Gaia_Sequoia_p_Thamnos_new_ratios['fe_h']
    galah_Gaia_GSE_new_ratios[X_H] = galah_Gaia_GSE_new_ratios[X_fe] + galah_Gaia_GSE_new_ratios['fe_h']
    Halo_Pradosh_galah_Gaia_new_ratios[X_H] = Halo_Pradosh_galah_Gaia_new_ratios[X_fe] + Halo_Pradosh_galah_Gaia_new_ratios['fe_h']
    galah_Gaia_Thamnos1_new_ratios[X_H] = galah_Gaia_Thamnos1_new_ratios[X_fe] + galah_Gaia_Thamnos1_new_ratios['fe_h']
    galah_Gaia_Thamnos2_new_ratios[X_H] = galah_Gaia_Thamnos2_new_ratios[X_fe] + galah_Gaia_Thamnos2_new_ratios['fe_h']
    
for X in element_list:
    
    cut_off = 0.10
    if X == 'fe':
        continue
    if X in ['y', 'ba', 'la', 'ce', 'nd', 'sm', 'eu']:
        cut_off = 0.12
    # print(X, cut_off)
    X_fe = X + '_fe'
    e_X_fe = 'e_' + X_fe
    galah_Gaia_new_ratios = galah_Gaia_new_ratios[galah_Gaia_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Sequoia_new_ratios = galah_Gaia_Sequoia_new_ratios[galah_Gaia_Sequoia_new_ratios[e_X_fe] < cut_off]
    Halo_Pradosh_galah_Gaia_new_ratios = Halo_Pradosh_galah_Gaia_new_ratios[Halo_Pradosh_galah_Gaia_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_GSE_new_ratios = galah_Gaia_GSE_new_ratios[galah_Gaia_GSE_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Halo_new_ratios = galah_Gaia_Halo_new_ratios[galah_Gaia_Halo_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Thamnos1_new_ratios = galah_Gaia_Thamnos1_new_ratios[galah_Gaia_Thamnos1_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Thamnos2_new_ratios = galah_Gaia_Thamnos2_new_ratios[galah_Gaia_Thamnos2_new_ratios[e_X_fe] < cut_off]
    galah_Gaia_Sequoia_p_Thamnos_new_ratios = galah_Gaia_Sequoia_p_Thamnos_new_ratios[galah_Gaia_Sequoia_p_Thamnos_new_ratios[e_X_fe] < cut_off]
    print(f'{X}, {cut_off}: {len(galah_Gaia_Sequoia_new_ratios)}')
print(f'Seq_lim: {len(galah_Gaia_Sequoia_new_ratios)}')
print(f'Seq_Tham: {len(galah_Gaia_Sequoia_p_Thamnos_new_ratios)}')
print(f'P_Halo: {len(Halo_Pradosh_galah_Gaia_new_ratios)}')
print(f'GSE: {len(galah_Gaia_GSE_new_ratios)}')
print(f'Halo: {len(galah_Gaia_Halo_new_ratios)}')
print(f'Thamnos1: {len(galah_Gaia_Thamnos1_new_ratios)}')
print(f'Thamnos2: {len(galah_Gaia_Thamnos2_new_ratios)}')
    

In [ ]:
''' THESE INCLUDE [ 'cu', 'ca', 'mn', 'ti','ba','na', 'nd'] and 4d '''
'''  GSE_vs_Sequoia and Halo_vs_GS_vs_Sequoia '''

# previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260520_012632_Galah_GSE_vs_Sequoia_vs_Halo_4d_cu_ca_mn_ti_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'

# previos_values_lcation_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260520_153001_Galah_Halo_vs_GS_vs_Sequoia_4d_cu_ca_mn_ti_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)

In [ ]:
''' THESE INCLUDE [ 'cu', 'ca', 'mn', 'ti', 'v', 'na', 'nd'] and 4d '''
'''  GSE_vs_Sequoia and Halo_vs_GS_vs_Sequoia '''

# previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260521_182738_Galah_GSE_vs_Sequoia_4d_cu_ca_mn_ti_V_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'

# previos_values_lcation_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260521_152626_Galah_Halo_vs_GSE_vs_Sequoia_4d_cu_ca_mn_ti_V_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)

In [ ]:
''' THESE INCLUDE [ 'cu', 'ca', 'mn', 'ti', 'v', 'ba', 'na', 'nd'] and 3d '''
'''  GSE_vs_Sequoia and Halo_vs_GS_vs_Sequoia '''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260522_134626_Galah_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'
previos_values_lcation_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)
p1_low = 0.875

previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260522_153935_Galah_Halo_vs_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'
previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)
p2_low = 0.765


In [ ]:
''' THESE INCLUDE [ 'cu', 'ca', 'mn', 'ti','v', 'ba', 'na', 'nd'] and 4d '''
'''  GSE_vs_Sequoia and Halo_vs_GS_vs_Sequoia and Halo_vs_Sequoia'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260523_180437_Galah_GSE_vs_Sequoia_4d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'
previos_values_lcation_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)
p1_low = 0.884

previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260523_092132_Galah_Halo_vs_GSE_vs_Sequoia_4d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'
previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)
p2_low = 0.771

# previos_values_lcation_Halo_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260526_074629_Galah_Halo_vs_Sequoia_4d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'
# previos_values_lcation_Halo_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_Sequoia)

# # first_Q = ((p_value_comparison['accuracy_GSE_Sequoia'] > 0.884) & (p_value_comparison['accuracy_all_3'] > 0.771))

In [ ]:
''' THESE INCLUDE [ 'cu', 'ca', 'mn', 'ti', 'v', 'ba', 'na', 'nd'] and 3d '''
''' THAMNOS Stars were not removed'''
'''  GSE_vs_Sequoia and Halo_vs_GS_vs_Sequoia '''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260527_154754_Galah_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75_Thamnos_Stars_kept/full_csv.csv'

previos_values_lcation_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260527_205216_Galah_Halo_vs_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75_Thamnos_Stars_kept/full_csv.csv'

previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)

In [ ]:
''' THESE INCLUDE [ 'cu', 'ca', 'mn', 'ti', 'v', 'ba', 'na', 'nd'] and 3d '''
''' I wanted to compare the distribution between the two times i ran these elements with and without the thamnos stars theoredically they should be simmilar'''

# previos_values_lcation_GSE_vs_Sequoia= '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260522_134626_Galah_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'

# previos_values_lcation_GSE_vs_Sequoia_raw_no_thamnos  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)

# previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260527_154754_Galah_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75_Thamnos_Stars_kept/full_csv.csv'

# previos_values_lcation_GSE_vs_Sequoia_raw_with_Thamnos = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)


# previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260522_153935_Galah_Halo_vs_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75/full_csv.csv'

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw_no_thamnos = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260527_205216_Galah_Halo_vs_GSE_vs_Sequoia_3d_cu_ca_mn_ti_v_ba_na_and_nd_also_fe_is_lessthan_minus0.75_Thamnos_Stars_kept/full_csv.csv'

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw_with_thamnos = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)

In [ ]:
''' THESE INCLUDE [ 'cu', 'ca', 'mn', 'ti', 'sc', 'v', 'na', 'nd'] and 3d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo_vs_GS_vs_Sequoia and Halo vs Sequoia'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260602_140401_Galah_GSE_vs_Sequoia_3d_cu_ca_mn_ti_sc_v_na_and_nd_also_fe_is_lessthan_minus0.75_Thamnos_Stars_removed/full_csv.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)
p1_low = 0.88

# previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260603_123825_Galah_Halo_vs_GSE_vs_Sequoia_3d_cu_ca_mn_ti_sc_v_na_and_nd_also_fe_is_lessthan_minus0.75_Thamnos_Stars_removed/full_csv.csv'
# previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)
# p2_low = 0.765

''' REALY THIS IS HALO VS SEQUOIA BUT I DIDNT WANT TO REWRITE ALL THE VARIABLES'''
previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260605_120535_Galah_Halo_vs_Sequoia_3d_cu_ca_mn_ti_sc_v_na_and_nd_also_fe_is_lessthan_minus0.75_Thamnos_Stars_removed/full_csv.csv'
previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)
p2_low = 0.964



In [ ]:
''' THESE INCLUDE elements_in_Pradosh_Galah_1 = ['ca', 'ti', 'ni','zr', 'ce', 'nd'] and 3d '''
''' THAMNOS Stars were removed'''
'''  GSE_vs_Sequoia and Halo vs Sequoia and Halo vs GSE'''

previos_values_lcation_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260626_144743_Galah_GSE_vs_Sequoia_3d_ca_ti_ni_zr_ce_nd_also_fe_is_lessthan_minus0.75_galah_phase_1_2_Hermes/full_csv.csv'
previos_values_lcation_GSE_vs_Sequoia_raw  = pd.read_csv(previos_values_lcation_GSE_vs_Sequoia)
p1_low = 0.85

''' REALY THIS IS HALO VS SEQUOIA BUT I DIDNT WANT TO REWRITE ALL THE VARIABLES'''
previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260626_145557_Galah_Halo_vs_Sequoia_3d_ca_ti_ni_zr_ce_nd_also_fe_is_lessthan_minus0.75_galah_phase_1_2_Hermes/full_csv.csv'
previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)
# p2_low = 0.964

''' REALY THIS IS HALO VS GSE BUT I DIDNT WANT TO REWRITE ALL THE VARIABLES'''
previos_values_lcation_Halo_vs_GSE_vs_Sequoia = '/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots/20260626_151106_Galah_Halo_vs_GSE_3d_ca_ti_ni_zr_ce_nd_also_fe_is_lessthan_minus0.75_galah_phase_1_2_Hermes/full_csv.csv'
previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw = pd.read_csv(previos_values_lcation_Halo_vs_GSE_vs_Sequoia)
p2_low = 0.77

In [ ]:
'''Oct 15 2026: I have not looked at this code in a while but I wanted to analize the RFC pltos again.
This is for elm list 18 so it has 12 elements.
It has SL v PHalo, ST v PHalo, SL v G, ST v G and G v PHalo.'''


In [ ]:
# match_cols = ['ratio1', 'ratio2', 'ratio3']
# # match_cols = ['ratio1', 'ratio2', 'ratio3', 'ratio4']

# previos_values_lcation_Halo_vs_Sequoia_raw_sorted = (
#     previos_values_lcation_Halo_vs_Sequoia_raw
#     .set_index(match_cols)
#     .loc[previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw.set_index(match_cols).index]
#     .reset_index()
# )

# p_value_comparison = previos_values_lcation_Halo_vs_Sequoia_raw_sorted.merge(
#     previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw[match_cols + ['accuracy']],
#     on=match_cols,
#     suffixes=('_Halo_Sequoia', '_all_3')
# )

# print(p_value_comparison)

# first_Q = ((p_value_comparison['accuracy_Halo_Sequoia'] > 0.964) & (p_value_comparison['accuracy_all_3'] > 0.771))

In [ ]:
''' This cell was made so I could compare the times when I included thamnos and when I didn't include thamnos'''

# match_cols = ['ratio1', 'ratio2', 'ratio3']
# match_cols = ['ratio1', 'ratio2', 'ratio3', 'ratio4']

# previos_values_lcation_GSE_vs_Sequoia_raw_sorted_no_thamnos = (
#     previos_values_lcation_GSE_vs_Sequoia_raw_no_thamnos
#     .set_index(match_cols)
#     .loc[previos_values_lcation_GSE_vs_Sequoia_raw_with_Thamnos.set_index(match_cols).index]
#     .reset_index()
# )

# p_value_comparison = previos_values_lcation_GSE_vs_Sequoia_raw_sorted_no_thamnos.merge(
#     previos_values_lcation_GSE_vs_Sequoia_raw_with_Thamnos[match_cols + ['accuracy']],
#     on=match_cols,
#     suffixes=('_GSE_Sequoia', '_all_3')
# )


# previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw_sorted_no_thamnos = (
#     previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw_no_thamnos
#     .set_index(match_cols)
#     .loc[previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw_with_thamnos.set_index(match_cols).index]
#     .reset_index()
# )

# p_value_comparison = previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw_sorted_no_thamnos.merge(
#     previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw_with_thamnos[match_cols + ['accuracy']],
#     on=match_cols,
#     suffixes=('_GSE_Sequoia', '_all_3')
# )

# print(p_value_comparison)

In [ ]:
match_cols = ['ratio1', 'ratio2', 'ratio3']
# match_cols = ['ratio1', 'ratio2', 'ratio3', 'ratio4']

previos_values_lcation_GSE_vs_Sequoia_raw_sorted = (
    previos_values_lcation_GSE_vs_Sequoia_raw
    .set_index(match_cols)
    .loc[previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw.set_index(match_cols).index]
    .reset_index()
)

p_value_comparison = previos_values_lcation_GSE_vs_Sequoia_raw_sorted.merge(
    previos_values_lcation_Halo_vs_GSE_vs_Sequoia_raw[match_cols + ['accuracy']],
    on=match_cols,
    suffixes=('_GSE_Sequoia', '_all_3')
    # suffixes=('_G_vs_S', '_H_vs_G')
)

print(p_value_comparison)

In [ ]:
first_Q = ((p_value_comparison['accuracy_GSE_Sequoia'] > p1_low) & (p_value_comparison['accuracy_all_3'] > p2_low) 
            # & ~((p_value_comparison['accuracy_GSE_Sequoia'] < 0.885) & (p_value_comparison['accuracy_all_3'] < 0.966))
            )

# S_and_A_rank_visual_ratios = ['mn_ca', 'nd_cu', 'v_ti', 'ti_mn', 'v_ca', 'na_mn', 'ca_cu', 'nd_na', 'nd_ca', 'nd_v'] # good without thamnos stars
# ranked_mask = (
#     p_value_comparison['ratio1'].isin(S_and_A_rank_visual_ratios)
#     & p_value_comparison['ratio2'].isin(S_and_A_rank_visual_ratios)
#     & p_value_comparison['ratio3'].isin(S_and_A_rank_visual_ratios)
#     # & p_value_comparison['ratio4'].isin(S_and_A_rank_visual_ratios)
# )


S_and_A_rank_visual_ratios = ['mn_ca', 'nd_cu', 'v_ti', 'ti_mn', 'v_ca', 'na_mn','nd_ti', 'nd_na', 'nd_ca', 'nd_v', 'mn_cu', 'nd_ba'] # good with thamnos stars
ranked_mask = (
    p_value_comparison['ratio1'].isin(S_and_A_rank_visual_ratios)
    & p_value_comparison['ratio2'].isin(S_and_A_rank_visual_ratios)
    & p_value_comparison['ratio3'].isin(S_and_A_rank_visual_ratios)
    # & p_value_comparison['ratio4'].isin(S_and_A_rank_visual_ratios)
)

In [ ]:
plt.scatter(p_value_comparison['accuracy_all_3'], p_value_comparison['accuracy_GSE_Sequoia'], c  = 'b', label  = 'All')
plt.scatter(p_value_comparison[first_Q]['accuracy_all_3'], p_value_comparison[first_Q]['accuracy_GSE_Sequoia'], c  = 'r', label = 'first Q')
# plt.scatter(p_value_comparison[ranked_mask]['accuracy_all_3'], p_value_comparison[ranked_mask]['accuracy_GSE_Sequoia'], c  = 'g', label = 'visual')

# plt.scatter(p_value_comparison['accuracy_all_3'], p_value_comparison['accuracy_Halo_Sequoia'], c  = 'b', label  = 'first Q')
# plt.scatter(p_value_comparison[first_Q]['accuracy_all_3'], p_value_comparison[first_Q]['accuracy_Halo_Sequoia'], c  = 'r', label = 'all')


x = np.array(p_value_comparison['accuracy_all_3'])
y = np.array(p_value_comparison['accuracy_GSE_Sequoia'])
# fit line
m, b = np.polyfit(x, y, 1)
# line values
y_fit = m * x + b
plt.plot(x, y_fit, label = 'best fit for all', c  = 'black', linewidth = 2)

plt.xlabel('all_three')
plt.ylabel('G_vs_S')
plt.legend()

In [ ]:
p_value_comparison[first_Q][0:-1]

In [ ]:
p_value_comparison['ratio 1'] = p_value_comparison['ratio1']
p_value_comparison['ratio 2'] = p_value_comparison['ratio2']
p_value_comparison['ratio 3'] = p_value_comparison['ratio3']

In [ ]:
high_acc_mask_table = ratio_totals_table(p_value_comparison[first_Q], galah_gaia_new_element_ratio_list, 3)
high_acc_mask_table

# high_acc_mask = (previos_values_lcation_GSE_vs_Sequoia_raw['accuracy'] > 0.885)
# high_acc_mask_table  = ratio_totals_table(previos_values_lcation_GSE_vs_Sequoia_raw[high_acc_mask], galah_gaia_new_element_ratio_list, 3)
# high_acc_mask_table

In [ ]:
p_value_comparison[ranked_mask][first_Q]

In [ ]:
''' MINI CORNER PLOTS '''

# e_labels = []
# for i in labels:
#     e_labels.append('e_' + i)
# sample_set = p_value_comparison[first_Q][ranked_mask]
sample_set = p_value_comparison[first_Q]
num = len(sample_set)
print(num)
acc_r = 'accuracy_GSE_Sequoia'
acc_l = 'accuracy_all_3'


TOP_50_ = top_min_corner_plots(sample_set, num)
TOP_lables = min_corner_plot_lables_comparison(sample_set, num, acc_r, acc_l)
parent_folder = "/home/simon/MQ NGC3201 project copy/MQ NGC3201 project copy/plots"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_folder = os.path.join(
    parent_folder,
    f"{timestamp}_Galah_scatter_comparison_for_3d_ca_ti_ni_zr_ce_and_nd_also_fe_is_lessthan_minus0.75_between_G_v_S_and_H_v_G_no_vand_hlines_W_Thamnos_Stars_removed"
)                                      # This contains the name of the folder
os.makedirs(save_folder, exist_ok=True)

# Save sample_set to the same folder
sample_set.to_csv(
    os.path.join(save_folder, "full_csv.csv"),
    index=False
)
for i in range(len(TOP_lables[0])):
    labels = TOP_lables[0][i]
    e_labels = TOP_lables[1][i]
    accuracy_1 = TOP_lables[2][i]
    accuracy_2 = TOP_lables[3][i]

    first_plot = 'galah_Gaia_Halo' #pick one of the keys in corner_datasets
    n_components = 2
    vline_and_hline = False

    plot_datapoints=True
    plot_contours=False
    fill_contours=False
    plot_density=False

    corner_datasets = {
        "galah_Gaia_Halo": (galah_Gaia_Halo_new_ratios[labels], galah_Gaia_Halo_new_ratios[e_labels],"grey", 1, 0.8),
        # "galah_Thickdisk": (thickdisk_galah_Gaia[labels], thickdisk_galah_Gaia[e_labels],"purple", 2.5, 1.0),
        "galah_Gaia_GSE": (galah_Gaia_GSE_new_ratios[labels], galah_Gaia_GSE_new_ratios[e_labels],"green", 2.0, 1.0),
        "galah_Gaia_Sequoia": (galah_Gaia_Sequoia_new_ratios[labels], galah_Gaia_Sequoia_new_ratios[e_labels],"red", 2.5, 1.0),
        # "galah_NGC5139": (galah_Gaia_NGC5139_new_ratios[labels], galah_Gaia_NGC5139_new_ratios[e_labels],"blue", 2.5, 1.0),
        # "galah_NGC3201": (galah_Gaia_NGC3201_new_ratios[labels], galah_Gaia_NGC3201_new_ratios[e_labels],"purple", 2.5, 1.0),
    }

    # keys = sorted(corner_datasets.keys())  # sorted = consistent colors across runs
    # cmap = plt.cm.tab10
    # colors = cmap(np.linspace(0, 0.8, len(keys)))

    # for key, color in zip(keys, colors):
    #     df_values, df_errors, _, ms, alpha = corner_datasets[key]
    #     corner_datasets[key] = (df_values, df_errors, color, ms, alpha)
    # # Convert all to values and keep data info

    for key in corner_datasets:
        df_values, df_errors, color, ms, alpha = corner_datasets[key] #data frame, color, markersize, alpha

        vals = df_values.to_numpy()
        # vals, mask = filter_constant_columns(vals)
        errs = df_errors.to_numpy()
        row_mask = np.any(np.isfinite(vals), axis=1)
        vals = vals[row_mask]
        errs = errs[row_mask]
        print(f"{key}: {df_values.shape[0]} stars, {df_values.shape[1]} element ratios")
        corner_datasets[key] = (vals, errs, color, ms, alpha)
    
    first_values, first_errors, first_color, first_ms, first_alpha = corner_datasets[first_plot]

        # Compute reasonable plotting ranges using percentiles
    ranges = []

    for dim in range(first_values.shape[1]):

        # combine all datasets for consistent limits
        combined = []

        for key, (values, errors, color, ms, alpha) in corner_datasets.items():
            combined.append(values[:, dim])

        combined = np.concatenate(combined)

        # remove NaNs/infs
        combined = combined[np.isfinite(combined)]

        # percentile clipping
        low = np.percentile(combined, 3)
        high = np.percentile(combined, 97)

        # optional padding
        padding = 0.05 * (high - low)

        ranges.append((low - padding, high + padding))

    fig = corner.corner(
    first_values,
    labels=labels,
    color=first_color,
    range = ranges,
    data_kwargs={'ms': first_ms, 'alpha': first_alpha, 'color': first_color},
    hist_kwargs={"density": True, "histtype": "step", "linewidth": 0.5},
    plot_datapoints=plot_datapoints,
    plot_contours=plot_contours,
    fill_contours=fill_contours,
    plot_density=plot_density,
    )
    ACC_1 = round(accuracy_1, 4)
    ACC_2 = round(accuracy_2, 4)
    fig.suptitle(f'Accuracy: {ACC_1} and {ACC_2}', fontsize=16, y=0.98)
    
    ndim = first_values.shape[1]
    axes = np.array(fig.axes).reshape((ndim, ndim))


    def overlay_gaussian(values, errors, color, n_components):
        for i in range(ndim):
            ax = axes[i, i]
            x = values[:, i]
            sigma_x = errors[:, i]

            mask = np.isfinite(x) & np.isfinite(sigma_x)
            x = x[mask]

            if len(x) > 5:  # need enough points
                # Fit 2-component Gaussian Mixture
                gmm = GaussianMixture(n_components=n_components)
                gmm.fit(x.reshape(-1, 1))

                x_grid = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1], 500)

                pdf = np.zeros_like(x_grid)



                # Sum the two Gaussians
                for weight, mean, cov in zip(gmm.weights_, gmm.means_, gmm.covariances_):
                    sigma = np.sqrt(cov[0][0])
                    pdf += weight * norm.pdf(x_grid, mean[0], sigma)

                    # (optional) plot individual components
                    ax.plot(x_grid,
                            weight * norm.pdf(x_grid, mean[0], sigma),
                            color=color,
                            lw=1,
                            # ls='dashed',
                            ls = "none",
                            alpha=0.7)

                # Plot total mixture
                ax.plot(x_grid, pdf, color=color, lw=0.5)
                

                #These next couple things will plot the verticle and horizontal lines
                if vline_and_hline == True:
                    max_index = np.argmax(pdf)
                    peak_x = x_grid[max_index]
                    peak_y = pdf[max_index]

                    print("Peak x:", peak_x)
                    print("Peak pdf:", peak_y)

                    # ax.axvline(peak_x, color=color, lw=1)
                    for row in range(ndim):

                        if row >= i:

                            ax_col = axes[row, i]

                            ax_col.axvline(
                                peak_x,
                                color=color,
                                lw=1,
                                alpha=0.8
                            )
                    
                    # draw horizontal line through entire row
                    for col in range(ndim):

                        # lower triangle exists
                        if i >= col:

                            ax_row = axes[i, col]

                            ax_row.axhline(
                                peak_x,
                                color=color,
                                lw=1,
                                alpha=0.8
                            )

    # Overlay remaining datasets
    handles = []
    for key, (values, errors, color, ms, alpha) in corner_datasets.items():
        if key != first_plot:
            corner.corner(
                values,
                fig=fig,
                color=color,
                range = ranges,
                data_kwargs={'ms': ms, 'alpha': alpha, 'color': color},
                hist_kwargs={"density": True, "histtype": "step", "linewidth": 0.5},
                plot_datapoints=plot_datapoints,
                plot_contours=plot_contours,
                fill_contours=fill_contours,
                plot_density=plot_density,
                show_titles=(key == "NGC3201"),
            )
            
        overlay_gaussian(values, errors, color, n_components)
        handles.append(plt.Line2D([0], [0], color = color, label = f'{key} N: {values.shape[0]}'))

    fig.legend(handles=handles,
            loc="upper right",
            bbox_to_anchor=(1.0, 0.90),
            #    ncol=len(handles),
            #    frameon=False,
            fontsize=12)
    
    plt.savefig(os.path.join(save_folder, f"plot_{i}_{labels}.png"))

    plt.close()

In [ ]:
labels = ['nd_cu', 'ba_v', 'na_v', 'na_ba']
labels = ['ti_cu', 'nd_cu', 'na_v', 'na_ba'] 

hbvbse = p_value_comparison[(p_value_comparison['ratio1'] == labels[0])
                            & (p_value_comparison['ratio2'] == labels[1])
                            & (p_value_comparison['ratio3'] == labels[2])
                            & (p_value_comparison['ratio4'] == labels[3])
                            ]

In [ ]:
# labels = ['ca_cu', 'ba_cu', 'mn_ca']
labels = ['ca_cu', 'mn_ca', 'ba_ca'] #BEST for when Sequoia is more different from the Halo than the GSE
# labels = ['ca_cu', 'v_ca', 'ba_ca']
labels = ['ca_cu', 'ba_ca', 'v_ti'] # 2nd best for when Sequoia is more different from the Halo than the GSE
# labels = ['ba_cu', 'mn_ca', 'ba_ca']
# labels = ['ba_cu', 'ba_ca', 'v_ti']

# labels = ['mn_ca', 'mn_ca', 'v_ti'] 
# labels = [ 'mn_ca', 'ti_mn','v_ti'] # This is a good one with and without thamnos stars
labels = ['mn_ca', 'v_ca', 'nd_ba'] # this is the best with thamnos stars

labels = ['ca_cu', 'v_ca', 'ti_mn']
labels = ['ti_ca', 'ce_ti', 'nd_zr']


# hbvbse = p_value_comparison[(p_value_comparison['ratio1'] == 'ti_cu')
#                             & (p_value_comparison['ratio2'] == 'na_mn')
#                             & (p_value_comparison['ratio3'] == 'v_ti')
#                             ]

hbvbse = p_value_comparison[(p_value_comparison['ratio1'] == labels[0])
                            & (p_value_comparison['ratio2'] == labels[1])
                            & (p_value_comparison['ratio3'] == labels[2])
                            ]

In [ ]:
hbvbse

In [ ]:
''' MINI CORNER PLOT FOR SPESIFIC GROUP '''

# labels = ['ca_ti', 'ti_ce', 'nd_zr']
e_labels = []
for i in labels:
    e_labels.append('e_' + i)

# for i in range(len(TOP_lables[0])):
    # labels = TOP_lables[0][i]
    # e_labels = TOP_lables[1][i]
    # accuracy_1 = TOP_lables[2][i]
    # accuracy_2 = TOP_lables[3][i]


accuracy_1 = hbvbse['accuracy_GSE_Sequoia'].iloc[0]
accuracy_2 = hbvbse['accuracy_all_3'].iloc[0]
    # accuracy_1 = hbvbse.iloc[0, -1]   # last column
# accuracy_2 = hbvbse.iloc[0, -2]   # second to last column

first_plot = 'Halo_Pradosh_galah' #pick one of the keys in corner_datasets
n_components = 2
vline_and_hline = False

plot_datapoints= True
plot_contours=True
fill_contours=True
plot_density=False

corner_datasets = {
    # "galah_Halo": (galah_Gaia_Halo_new_ratios[labels], galah_Gaia_Halo_new_ratios[e_labels],"grey", 1, 0.8),
    "Halo_Pradosh_galah": (Halo_Pradosh_galah_Gaia[labels], Halo_Pradosh_galah_Gaia[e_labels],"grey", 1, 0.8),

    # "galah_Thickdisk": (thickdisk_galah_Gaia[labels], thickdisk_galah_Gaia[e_labels],"purple", 2.5, 1.0),
    "GSE": (galah_Gaia_GSE_new_ratios[labels], galah_Gaia_GSE_new_ratios[e_labels],"green", 2.0, 1.0),
    "Sequoia": (galah_Gaia_Sequoia_new_ratios[labels], galah_Gaia_Sequoia_new_ratios[e_labels],"red", 2.5, 1.0),
    # "galah_NGC5139": (galah_Gaia_NGC5139_new_ratios[labels], galah_Gaia_NGC5139_new_ratios[e_labels],"blue", 2.5, 1.0),
    # "galah_NGC3201": (galah_Gaia_NGC3201_new_ratios[labels], galah_Gaia_NGC3201_new_ratios[e_labels],"purple", 2.5, 1.0),
    
}

# keys = sorted(corner_datasets.keys())  # sorted = consistent colors across runs
# cmap = plt.cm.tab10
# colors = cmap(np.linspace(0, 0.8, len(keys)))

# for key, color in zip(keys, colors):
#     df_values, df_errors, _, ms, alpha = corner_datasets[key]
#     corner_datasets[key] = (df_values, df_errors, color, ms, alpha)
# # Convert all to values and keep data info

for key in corner_datasets:
    df_values, df_errors, color, ms, alpha = corner_datasets[key] #data frame, color, markersize, alpha

    vals = df_values.to_numpy()
    # vals, mask = filter_constant_columns(vals)
    errs = df_errors.to_numpy()
    row_mask = np.any(np.isfinite(vals), axis=1)
    vals = vals[row_mask]
    errs = errs[row_mask]
    print(f"{key}: {df_values.shape[0]} stars, {df_values.shape[1]} element ratios")
    corner_datasets[key] = (vals, errs, color, ms, alpha)

first_values, first_errors, first_color, first_ms, first_alpha = corner_datasets[first_plot]

    # Compute reasonable plotting ranges using percentiles
ranges = []

for dim in range(first_values.shape[1]):

    # combine all datasets for consistent limits
    combined = []

    for key, (values, errors, color, ms, alpha) in corner_datasets.items():
        combined.append(values[:, dim])

    combined = np.concatenate(combined)

    # remove NaNs/infs
    combined = combined[np.isfinite(combined)]

    # percentile clipping
    low = np.percentile(combined, 1)
    high = np.percentile(combined, 99)

    # optional padding
    padding = 0.05 * (high - low)

    ranges.append((low - padding, high + padding))

fig = corner.corner(
first_values,
labels=labels,
color=first_color,
range = ranges,
data_kwargs={'ms': first_ms, 'alpha': first_alpha, 'color': first_color},
hist_kwargs={"density": True, "histtype": "step", "linewidth": 0.5},
plot_datapoints=plot_datapoints,
plot_contours=plot_contours,
fill_contours=fill_contours,
plot_density=plot_density,
# plot_datapoints=False,
# plot_contours=True,
# fill_contours=True,
# plot_density=True,

contour_kwargs={"linewidths": 0.4, "alpha": 1},
fill_contour_kwargs={"alpha": 0.1},
)

ACC_1 = round(accuracy_1, 4)
print(accuracy_1)
ACC_2 = round(accuracy_2, 4)
fig.suptitle(f'Accuracy: {ACC_1} and {ACC_2}', fontsize=16, y=0.98)

ndim = first_values.shape[1]
axes = np.array(fig.axes).reshape((ndim, ndim))

def overlay_gaussian(values, errors, color, n_components):
    for i in range(ndim):
        ax = axes[i, i]
        x = values[:, i]
        sigma_x = errors[:, i]

        mask = np.isfinite(x) & np.isfinite(sigma_x)
        x = x[mask]

        if len(x) > 5:  # need enough points
            # Fit 2-component Gaussian Mixture
            gmm = GaussianMixture(n_components=n_components)
            gmm.fit(x.reshape(-1, 1))

            x_grid = np.linspace(ax.get_xlim()[0], ax.get_xlim()[1], 500)

            pdf = np.zeros_like(x_grid)

            # Sum the two Gaussians
            for weight, mean, cov in zip(gmm.weights_, gmm.means_, gmm.covariances_):
                sigma = np.sqrt(cov[0][0])
                pdf += weight * norm.pdf(x_grid, mean[0], sigma)

                # (optional) plot individual components
                ax.plot(x_grid,
                        weight * norm.pdf(x_grid, mean[0], sigma),
                        color=color,
                        lw=1,
                        # ls='dashed',
                        ls = "none",
                        alpha=0.7)

            # Plot total mixture
            ax.plot(x_grid, pdf, color=color, lw=0.5)

                #These next couple things will plot the verticle and horizontal lines
            if vline_and_hline == True:
                max_index = np.argmax(pdf)
                peak_x = x_grid[max_index]
                peak_y = pdf[max_index]

                print("Peak x:", peak_x)
                print("Peak pdf:", peak_y)

                # ax.axvline(peak_x, color=color, lw=1)
                for row in range(ndim):

                    if row >= i:

                        ax_col = axes[row, i]

                        ax_col.axvline(
                            peak_x,
                            color=color,
                            lw=1,
                            alpha=0.8
                        )
                
                # draw horizontal line through entire row
                for col in range(ndim):

                    # lower triangle exists
                    if i >= col:

                        ax_row = axes[i, col]

                        ax_row.axhline(
                            peak_x,
                            color=color,
                            lw=1,
                            alpha=0.8
                        )

# Overlay remaining datasets
handles = []
for key, (values, errors, color, ms, alpha) in corner_datasets.items():
    if key != first_plot:
        corner.corner(
            values,
            fig=fig,
            color=color,
            range = ranges,
            data_kwargs={'ms': ms, 'alpha': alpha, 'color': color},
            hist_kwargs={"density": True, "histtype": "step", "linewidth": 0.5},
            plot_datapoints=plot_datapoints,
            plot_contours=plot_contours,
            fill_contours=fill_contours,
            plot_density=plot_density,
            contour_kwargs={"linewidths": 0.4, "alpha": 0.8},
            # fill_contour_kwargs={"alpha": 0.1},
            show_titles=(key == "NGC3201"),
        )
    overlay_gaussian(values, errors, color, n_components)
    handles.append(plt.Line2D([0], [0], color = color, label = f'{key} N: {values.shape[0]}'))

fig.legend(handles=handles,
        loc="upper right",
        bbox_to_anchor=(1.0, 0.90),
        #    ncol=len(handles),
        #    frameon=False,
        fontsize=12)


In [ ]:
# ratio_list = galah_gaia_new_element_ratio_list
# ratio_list = S_and_A_rank_visual_ratios
ratio_list = high_acc_mask_table['ratio'].to_list()

n_components = 2


object_list = {
                'GSE': (galah_Gaia_GSE_new_ratios, 'g', 1, 0.5, 0.02),  #df, color, linewidth, alpha, shift_right
                'Sequoia': (galah_Gaia_Sequoia_new_ratios, 'r', 1, 0.5, 0.10),
                'Halo': (galah_Gaia_Halo_new_ratios, 'grey', 1, 0.5, 0.18)
}



for ratio in ratio_list:
    plt.figure(figsize=(10, 6))
    all_values = []

    for key, (df, color, lw, alpha, shift_right) in object_list.items():

        values = df[ratio].to_numpy()
        values = values[np.isfinite(values)]
        all_values.append(values)

        df_copy = df
        num_stars = str(len(df_copy))
        plot_gmm(df_copy[ratio] , f'{key}: N = {num_stars}', color, n_components, maximum_peak = True, shift_right = shift_right)
        plt.hist(df[ratio], density = True, bins  = 30, histtype='step', facecolor = 'none', alpha = alpha, edgecolor=color, linewidth = lw)
        # mean = np.round(np.mean(df[ratio]), 3)
        # plt.axvline(x=mean, color=color, linestyle='--')

    all_values = np.concatenate(all_values)
    low = np.percentile(all_values, 1)
    high = np.percentile(all_values, 99)

    plt.xlim(low, high)

    plt.title(F'Galah {ratio} distribution')
    plt.xlabel(f'[{ratio}]')
    plt.ylabel('density')
    plt.legend()